# Day 09 · 멀티에이전트 오케스트레이션

8강에서 **한 명**을 제대로 부렸다. 오늘은 **팀**을 만든다 — 그리고 그 팀을 **내가 짓는다**.

> 받아 쓰면 빠르지만 **블랙박스**다. 오늘은 그 구조를 직접 짓는다.

**오늘의 실습 2개**

| 랩 | 무엇을 | 결과물 |
|---|---|---|
| Lab 1 | **내 오케스트레이터 짓기** | orchestrate · 서브에이전트 4 · hook · 플러그인 |
| Lab 2 | **업무 관리 툴** (메인) | 위임하고 게이트로 검증한 동작하는 앱 |

**판단 3문** — 멀티로 갈지 먼저 가린다.
① 병렬 분해가 가능한가? ② 가치가 **15배 토큰**보다 큰가? ③ 사람이 검수 가능한 단위인가?
**하나라도 No → 단일 + Plan Mode.**


## Lab 1 · 내 오케스트레이터 짓기

먼저 기성품을 설치해 **어떻게 움직이는지 관찰**하고, 그다음 **같은 구조를 직접 짓는다**.

### STEP 0 · 기성품 먼저 (관찰용)

```
# Superpowers — 방법론을 코드로 강제
/plugin marketplace add obra/superpowers-marketplace
/plugin install superpowers@superpowers-marketplace

# gstack — 가상 개발 팀 (역할 페르소나)
git clone https://github.com/garrytan/gstack.git ~/.claude/skills/gstack && cd ~/.claude/skills/gstack && ./setup
```

받아 쓰면 빠르지만 블랙박스다 — 이제 직접 만든다.

---

### LAB A · orchestrate — 리드를 만든다

Claude Code에 **그대로 입력**한다:

```
~/.claude/skills/orchestrate/SKILL.md 만들어줘

이런 리드 오케스트레이터가 필요해:
· name: orchestrate · description엔 발동조건 한 문장
· "○○ 기능 만들어줘" 하면 발동 · 입력은 $ARGUMENTS
· 직접 코드 쓰지 말고 이 순서로 강제 위임 — 건너뛰기 금지:
  ① @planner 기획·분해 (인수조건 + 독립 chunk) → 사람 승인
  ② @coder 병렬 — chunk마다 worktree, 테스트 먼저(RED) → 구현(GREEN)
  ③ @reviewer 읽기전용 검수 (스펙 준수 + 코드 품질)
  ④ @tester 전체 검증 게이트
· 검증 실패 → 담당 에이전트로 되돌림
· "부탁"이 아니라 "순서"
```

**나오면 확인할 것** — 머리말(name + description = 트리거) / 본문(서브에이전트 · 순서 강제 · worktree · `$ARGUMENTS`)

> 한 줄 원칙: **리드는 위임만 한다. 직접 코딩 안 함. 실패는 되돌림.**

**worktree 병렬 격리** — coder를 병렬로 돌리려면 각자 격리된 작업 공간이 필요하다:

```bash
git worktree add .claude/worktrees/<id> -b chunk/<id>
# 그 안에서 RED→GREEN TDD · 타 chunk 안 건드림
git worktree remove .claude/worktrees/<id>
```

**읽기 공유 · 쓰기 분리.** 패턴 = **Supervisor + Fan-out**.

---

### LAB B · 역할별 서브에이전트 — 권한이 곧 경계

```
~/.claude/agents/ 에 네 개 만들어줘
각 .md: name · description(트리거) · tools · 역할 프롬프트

· planner:  계획만 — 수정 금지          (Read, Grep)
· coder:    자기 worktree에서만 쓰기     (Read, Write, Edit, Bash)
· reviewer: 읽기전용 — 수정·실행 금지    (Read, Grep, Glob)
· tester:   테스트 실행만 — 통과/실패 판정 (Read, Bash)
```

**tools가 곧 경계다** — reviewer는 못 고치고, coder는 worktree 밖을 못 건드린다.
**부탁은 무시돼도 권한은 못 뚫는다.**

---

### LAB B · .env 차단 hook — 마지막 게이트

**왜?** coder는 쓰기 권한이 있어 `.env`도 건드릴 수 있고, 프롬프트 "부탁"은 무시될 수 있다.
hook = **누가 시키든 도구 호출 직전에 코드로 막는** 마지막 게이트.

```javascript
// hooks/guard.mjs  (ESM — import, require 금지)
import { readFileSync } from 'node:fs';
const input = JSON.parse(readFileSync(0, 'utf8'));
const p = (input.tool_input?.file_path || '').replace(/\\/g, '/').toLowerCase();
const cmd = input.tool_input?.command || '';
const BLOCK = ['.env', '/.git/', 'id_rsa', 'secrets', '.pem'];

if (BLOCK.some(b => p.includes(b)) || /rm\s+-rf\s+[\/~]/.test(cmd)) {
  console.error('차단: 민감 경로 / 위험 명령');
  process.exit(2);   // 2 = 도구 호출 차단
}
process.exit(0);
```

```json
{ "hooks": { "PreToolUse": [{
  "matcher": "Read|Edit|Write|Bash",
  "hooks": [{ "type": "command", "command": "node ${CLAUDE_PLUGIN_ROOT}/hooks/guard.mjs" }]
}]}}
```

**단독 테스트:**
```bash
echo '{"tool_input":{"file_path":"x/.env"}}' | node harness/hooks/guard.mjs; echo $?   # → 2
```

---

### LAB C · 플러그인으로 묶기

```
harness/
├── .claude-plugin/plugin.json
├── skills/{orchestrate, spec, tdd}/SKILL.md
├── agents/{planner, coder, reviewer, tester}.md
└── hooks/{hooks.json, guard.mjs}
```

```bash
claude plugin validate ./harness
claude --plugin-dir ./harness
```

### 실행 — 직접 쓴 지휘자로 한 번에

```
/harness:orchestrate 작업 상태 전이 추가
  @planner → @coder(worktree 병렬, TDD) → @reviewer → @tester
```

`.env` 접근 시 hook이 **exit 2로 차단**하는지 확인한다.

> **오늘 손으로 지은 것** — orchestrate 리드 · 역할 서브에이전트 4 · worktree 병렬 · guard.mjs 게이트 · harness 플러그인


## Lab 2 · 업무 관리 툴 (메인 실습)

멀티에이전트는 **더 빨리 만드는 도구가 아니다** — 사람이 코드를 다 안 읽어도 되게 **위임**하고, **게이트로 검증·통제**하는 방식이다.

### 만들 것 — 작은 협업 보드

개념 **3가지**만 다룬다:
- **사람 (User)** — 등록자 · 담당자
- **할 일 (Task)** — 제목 · 내용 · 상태(대기→진행→완료) · 담당자
- **코멘트 (Comment)** — 할 일에 달리는 메모

> **작게 유지** — 기본 동작 5개 이내 · 한 문장으로 설명되고 · 남은 시간에 데모 가능. 그 이상은 안 만든다.

### Stage A · 보일러플레이트 받기

```bash
git clone <제공 레포> task-tool
cd task-tool && npm install
cp .env.example .env          # Windows: copy
npm run db:push && npm run db:seed
npm run dev                   # localhost:3000
```

Next.js · TypeScript · Prisma · **인증(JWT)·User 완성** · Vitest · AGENTS.md + CLAUDE.md 포함.
**셋업 0** — 멀티에이전트로 **Task·Comment 추가**에 바로 집중한다.

### Stage B · 우리 harness로 (Lab 1에서 지은 것)

**B-2 · 스펙 만들기**
```
/harness:spec
  "업무 관리 툴 — User·인증은 있음. Task·Comment 추가.
   상태 대기→진행→완료, 등록자/담당자 권한, 목록·상세·코멘트."

→ 빠진 요구사항을 질문으로 보완 (Socratic)
→ docs/prd.md 확정
```
> **prd.md가 정확할수록 orchestrate가 헛돌지 않는다** — 사양 한 줄이 구현 수십 줄을 좌우한다.

**B-3 · 작업 분해 (사람 승인 게이트)**
```
/harness:orchestrate "Task·Comment + 상태 전이"

@planner — prd.md·스키마 → .planning/IMPL_PLAN.md
  chunk 1 — Task CRUD + 상태 머신
  chunk 2 — 상태 전이 + 역할 권한
  chunk 3 — Comment + 목록/검색

사람 승인 대기 — 계획 확정 후 실행
```

**B-4 · 병렬 실행 → 검증**
```
chunk 1·2·3 → @coder 병렬 (각자 worktree, RED→GREEN)
@reviewer — 스펙 준수 + 코드 품질 (읽기전용)
@tester   — 테스트 실행 → 통과/실패
통과한 worktree만 병합
```

**B-5 · 사람 검증은 행동으로**
```bash
npm run dev
```
업무 등록 → 담당자 로그인 → 상태 변경(대기→진행→완료) → 코멘트

> **모르는 코드일수록 읽지 말고 클릭 사이클로 확인한다.**

### 기성 대안과의 매핑

| 단계 | 우리 harness | 기성 |
|---|---|---|
| 스펙 | `/harness:spec` | `/superpowers:brainstorm` · `/office-hours` |
| 계획 | `/harness:orchestrate` → @planner | `/superpowers:write-plan` · `/plan-eng-review` |
| 구현 | @coder 병렬 (worktree·TDD) | `/superpowers:execute-plan` |
| 검증 | @reviewer + @tester | `/review` · `/qa` |

구조를 알면 **갈아끼울 수 있다**.


## 산출물 체크리스트

- [ ] **판단 3문**으로 이 작업이 멀티에이전트 대상인지 가렸다
- [ ] Superpowers · gstack을 설치해 **어떻게 움직이는지 관찰**했다
- [ ] `orchestrate` 스킬을 **직접 만들었다** — 리드는 위임만
- [ ] `@planner/@coder/@reviewer/@tester` 를 **최소 권한으로** 만들었다
- [ ] **tools가 곧 경계**임을 확인했다 (reviewer는 못 고친다)
- [ ] `guard.mjs` 훅으로 **.env 접근을 exit 2로 차단**했다
- [ ] `harness/` 플러그인으로 **묶어서 배포 가능**하게 했다
- [ ] 업무 관리 툴을 **스펙 → 분해 → 승인 → 병렬 구현 → 검증**으로 만들었다
- [ ] 코드를 다 읽지 않고 **클릭 사이클로 검증**했다

> 한 줄: **모델은 계속 좋아진다. 하네스는 내가 만든다.**
